## 1. 核心思想

当用户提出**复杂问题**（如对比、多条件、多实体）时，单一的向量检索往往难以兼顾所有方面。子查询策略利用 LLM 将复杂问题**分解**为多个简单的子问题，分别进行检索，最后汇总上下文生成综合答案。

## 2. 核心逻辑伪代码
将“串行”的复杂问题转化为“并行”的简单检索，并进行去重（Deduplication）处理，这是该策略最关键的代码细节。

In [ ]:
def sub_query_rag_flow(complex_query):
    """
    核心流程：分解 -> 并行检索 -> 去重 -> 综合
    """

    # Step 1: LLM 分解 (One to Many)
    # 输入: "对比 A 和 B"
    # 输出: ["A的特征是什么?", "B的特征是什么?"]
    sub_queries = llm_decompose(complex_query)

    all_retrieved_docs = {} # 使用字典进行去重 (Key: DocID, Value: Content)

    # Step 2: 并行/循环检索 (Parallel Retrieval)
    for query in sub_queries:
        vector = embedding_model.encode(query)
        results = vector_db.search(vector, top_k=3)

        # Step 3: 结果聚合与去重 (Aggregation & Deduplication)
        for doc in results:
            if doc.id not in all_retrieved_docs:
                all_retrieved_docs[doc.id] = doc.text

    # Step 4: 综合生成 (Synthesis)
    # 将去重后的所有上下文合并，回答原始的复杂问题
    combined_context = "\n".join(all_retrieved_docs.values())
    final_answer = llm.generate(query=complex_query, context=combined_context)

    return final_answer

## 3. 完整实现（Milvus + OpenAI）

### 3.1 配置

In [ ]:
import os
from pymilvus import MilvusClient
from openai import OpenAI

# --- 配置部分 ---
os.environ["OPENAI_API_KEY"] = "sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9"  # 替换你的 Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
client = OpenAI()
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")

COLLECTION_NAME = "tech_docs"
DIMENSION = 1536

### 3.2 核心函数封装

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-small")
    return response.data[0].embedding

def llm_chat(prompt, model="gpt-4"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

### 3.3 准备milvus集合

In [ ]:
if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIMENSION,
    auto_id=True,
    enable_dynamic_field=True
)

### 3.4 构建索引

In [ ]:
# 模拟两类截然不同的文档，分别介绍 Milvus 和 Zilliz Cloud
docs = [
    # 关于 Milvus 的文档
    "Milvus 是一个开源的向量数据库，旨在为嵌入相似性搜索和 AI 应用程序提供支持。它支持多种索引类型（如 IVF、HNSW），需要用户自行部署和运维（如使用 Docker 或 K8s）。Milvus 专注于高性能和灵活性。",
    "Milvus 的架构包含接入层、协调服务、工作节点和存储层。它采用存储计算分离架构，适合有技术团队支持、希望完全掌控数据的企业。",
    # 关于 Zilliz Cloud 的文档
    "Zilliz Cloud 是基于 Milvus 构建的全托管向量数据库服务（SaaS）。它提供了开箱即用的体验，无需用户管理底层基础设施。",
    "Zilliz Cloud 提供了企业级特性，如自动扩缩容、存算分离的高级优化、SOC2 合规安全性以及专门的向量数据库管理控制台。它适合希望快速落地 AI 应用且不想运维基础设施的团队。"
]

print("正在构建索引...")
data = [{"vector": get_embedding(doc), "text": doc} for doc in docs]
print(f"插入 {len(docs)} 条文档。\n" + "-"*50)

### 3.5 插入milvus

In [ ]:
insert_result=milvus_client.insert(collection_name=COLLECTION_NAME, data=data)
print(f"\n成功插入 {insert_result['insert_count']} 条数据 。\n")

## 3.6. 检索
- 定义检索函数

In [ ]:
def decompose_query(original_query):
    """
    阶段一：查询分解
    让 LLM 将复杂问题拆解为独立且简单的子问题
    """
    prompt = f"""
    你是一个查询分析专家。请将以下复杂的用户查询分解为 2 到 4 个简单的、独立的子查询。
    这些子查询应该涵盖原始查询的所有方面，以便通过分别检索来回答原始问题。

    用户查询: "{original_query}"

    要求：
    1. 每个子查询占一行。
    2. 去掉序号，只输出问题文本。
    3. 子查询必须是完整的句子。
    """

    result = llm_chat(prompt)
    # 清洗输出，转为列表
    sub_queries = [line.strip() for line in result.split('\n') if line.strip()]
    return sub_queries

def search_and_deduplicate(sub_queries):
    """
    阶段二：并行检索与去重
    """
    unique_docs = {} # 使用 Dict {doc_id: doc_text} 去重

    print(f"开始执行 {len(sub_queries)} 个子查询检索...")

    for i, query in enumerate(sub_queries):
        print(f"  [{i+1}] 检索子查询: {query}")
        vector = get_embedding(query)

        results = milvus_client.search(
            collection_name=COLLECTION_NAME,
            data=[vector],
            limit=2, # 每个子问题只取 Top 2
            output_fields=["text"]
        )

        for res in results[0]:
            doc_id = res["id"]
            if doc_id not in unique_docs:
                unique_docs[doc_id] = res["entity"]["text"]
                # 打印日志方便观察
                print(f"      -> 命中新文档 (ID: {doc_id})")
            else:
                print(f"      -> 文档已存在，跳过 (ID: {doc_id})")

    return list(unique_docs.values())

def synthesize_answer(original_query, context_list):
    """
    阶段三：综合回答
    """
    context_str = "\n\n".join(context_list)

    prompt = f"""
    请根据以下参考文档，详细回答用户的问题。
    如果参考文档包含不同实体的信息，请进行对比分析。

    参考文档:
    {context_str}

    用户问题: "{original_query}"
    """

    return llm_chat(prompt, model="gpt-5.1") # 用更好的模型做总结

## 3.7 检索
- 测试

In [ ]:
user_query = "Milvus 和 Zilliz Cloud 在部署方式和适用场景上有什么区别？"

print(f"原始提问: {user_query}\n")

# 1. 分解
sub_queries = decompose_query(user_query)
print(f"LLM 分解后的子查询: {sub_queries}\n")

# 2. 检索 (包含去重)
retrieved_contexts = search_and_deduplicate(sub_queries)
print(f"\n最终聚合了 {len(retrieved_contexts)} 个唯一的文档块作为上下文。\n")

# 3. 生成
print("正在生成最终回答...\n")
final_answer = synthesize_answer(user_query, retrieved_contexts)

print("="*20 + " 最终 RAG 回答 " + "="*20)
print(final_answer)
print("="*55)

## 4. 使用场景

- 多维度查询："这辆车的安全性、价格和内饰怎么样？"（分解为三个独立的属性查询）。

- 长逻辑链：虽然不如 ReAct Agent 灵活，但对于单跳并列逻辑非常有效。